# 02 - Data Preprocessing and Unsupervised Classification

## Notebook Objective
Given the lack of ground truth labels regarding the health status of the glomeruli (e.g., healthy vs. necrotized), a supervised classification cannot be applied. Instead, this notebook implements a clustering pipeline consisting of three main phases:

1. **Data Standardization (Preprocessing):** Convolutional Neural Networks (CNNs) require uniform input tensors. Since the extracted patches have variable sizes, a padding technique is applied to resize all images to a standard dimension (e.g., 224x224 pixels) without altering the critical aspect ratio and morphological features of the tissue.
2. **Deep Feature Extraction:** A pre-trained deep learning model (e.g., ResNet or VGG) will be utilized as a feature extractor. The network will map the raw pixel data of each glomerulus into a high-dimensional feature vector representing its visual characteristics (textures, cellular density, sclerosis patterns).
3. **Dimensionality Reduction and Clustering:** The extracted high-dimensional features will be reduced using techniques such as PCA or UMAP. Finally, unsupervised clustering algorithms (e.g., K-Means or DBSCAN) will group the glomeruli into clusters based on their morphological similarities, aiming to implicitly separate the classes based on varying levels of necrotization

---

### Data Standardization: image loading and padding

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Define input parameters
dataset_dir = "./dataset_glomeruli_all"
target_size = (224, 224)

# Retrieve all image paths
image_paths = glob.glob(os.path.join(dataset_dir, "*.png"))
print(f"Total images found: {len(image_paths)}")

def resize_with_padding(img_path, expected_size):
    """
    Resizes an image maintaining the aspect ratio.
    Pads the remaining space with white pixels to match expected_size.
    """
    img = Image.open(img_path).convert("RGB")
    
    # Calculate the ratio to fit the image within target size
    img.thumbnail(expected_size, Image.Resampling.LANCZOS)
    
    # Create a new square white background
    background = Image.new("RGB", expected_size, (255, 255, 255))
    
    # Paste the resized image into the center of the background
    offset_x = (expected_size[0] - img.size[0]) // 2
    offset_y = (expected_size[1] - img.size[1]) // 2
    background.paste(img, (offset_x, offset_y))
    
    return background

# Process a batch of images to build the dataset array
processed_images = []
image_names = []

print("Processing and resizing images...")

for path in image_paths:
    # Apply padding and resizing
    padded_img = resize_with_padding(path, target_size)
    
    # Convert PIL Image to Numpy Array and normalize pixel values between 0 and 1
    img_array = np.array(padded_img) / 255.0
    
    processed_images.append(img_array)
    
    # Store the filename for future reference during clustering analysis
    file_name = os.path.basename(path)
    image_names.append(file_name)

# Convert lists to final numpy arrays
X_data = np.array(processed_images)
names_data = np.array(image_names)

print(f"Dataset shape: {X_data.shape}")
print("Data preprocessing complete.")

# Visualization check for the first processed image
plt.figure(figsize=(4, 4))
plt.imshow(X_data[0])
plt.title(f"Padded Image\n{names_data[0]}")
plt.axis("off")
plt.show()

### Deep Feature Extraction (using Keras ResNet50)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

print("Preparing data for ResNet50...")
# X_data is in [0, 1] range from previous cell. ResNet50 requires [0, 255] before preprocessing
X_data_raw = (X_data * 255.0).astype(np.float32)

# Apply ImageNet specific preprocessing (mean centering, RGB to BGR conversion)
X_preprocessed = preprocess_input(X_data_raw)

print("Loading pre-trained ResNet50...")
# Load model without classification head and add Global Average Pooling
feature_extractor = ResNet50(
    weights='imagenet', 
    include_top=False, 
    input_shape=(224, 224, 3), 
    pooling='avg'
)

print("Extracting features (Forward pass)...")
# Predict method processes the images in batches automatically
deep_features = feature_extractor.predict(X_preprocessed, batch_size=32)

print(f"Extraction complete.")
print(f"Original images shape: {X_data.shape}")
print(f"Deep features shape: {deep_features.shape}")

### Dimensionality Reduction via PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

print("Applying Principal Component Analysis (PCA)...")

# Initialize PCA to retain 95% of the variance, as seen in Lab 5
pca = PCA(n_components=0.95, random_state=42)

# Fit and transform the deep features
reduced_features = pca.fit_transform(deep_features)

print(f"Original feature dimension: {deep_features.shape[1]}")
print(f"Reduced feature dimension: {reduced_features.shape[1]}")

# Visualize the cumulative explained variance
plt.figure(figsize=(6, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
plt.title("PCA Cumulative Explained Variance")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Variance")
plt.grid(True)
plt.show()

### Unsupervised Clustering (K-Means) and Visualization

In [ ]:
from sklearn.cluster import KMeans
import random
import numpy as np
import matplotlib.pyplot as plt

# The PDF figure shows 6 progression stages (A through F)
n_clusters = 6

print(f"Applying K-Means clustering to find {n_clusters} clusters...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(reduced_features)

print("Clustering complete. Analyzing cluster distribution:")
# Count how many glomeruli ended up in each cluster
for i in range(n_clusters):
    count = np.sum(cluster_labels == i)
    print(f"  - Cluster {i}: {count} samples")

# --- VISUALIZATION OF THE CLUSTERS ---
# We want to plot a grid: 1 row for each cluster, showing 5 random images per cluster
samples_per_cluster = 5

fig, axes = plt.subplots(n_clusters, samples_per_cluster, figsize=(15, 3 * n_clusters))
fig.suptitle("K-Means Clustering Results (6 Stages of Necrotization)", fontsize=16)

for cluster_id in range(n_clusters):
    # Find the indices of all images belonging to the current cluster
    indices_in_cluster = np.where(cluster_labels == cluster_id)[0]
    
    # Select random samples to display (or all if there are fewer than samples_per_cluster)
    sample_indices = random.sample(
        list(indices_in_cluster), 
        min(samples_per_cluster, len(indices_in_cluster))
    )
    
    for i, idx in enumerate(sample_indices):
        # Handle cases where there might be fewer than 5 images in a cluster
        if len(sample_indices) == 0:
            continue
            
        ax = axes[cluster_id, i]
        
        # Display the original padded image
        ax.imshow(X_data[idx])
        ax.axis('off')
        
        # Add title only to the first image of each row
        if i == 0:
            ax.set_title(f"Cluster {cluster_id}\n{names_data[idx]}", fontsize=10, loc='left')
        else:
            ax.set_title(f"{names_data[idx]}", fontsize=8)

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

### Clustering Evaluation (Silhouette Score)

In [ ]:
from sklearn.metrics import silhouette_score

# Calculate Silhouette Score using the reduced features and the K-Means labels
sil_score = silhouette_score(reduced_features, cluster_labels)

print("--- Clustering Evaluation ---")
print(f"Silhouette Score: {sil_score:.4f}")

# Brief interpretation guide for the console output
if sil_score > 0.5:
    print("Interpretation: Strong cluster structure found. The 6 stages are well separated.")
elif sil_score > 0.2:
    print("Interpretation: Moderate cluster structure. The stages have some overlap, which is biologically expected.")
else:
    print("Interpretation: Weak cluster structure. The network struggles to separate the progression stages clearly.")

### Manifold Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print("Computing t-SNE to visualize the high-dimensional feature space in 2D...")
print("(This may take a minute depending on dataset size)")

# Initialize t-SNE
# Perplexity is a crucial parameter; 30 is a standard default for medical datasets
tsne = TSNE(n_components=2, perplexity=30.0, random_state=42)

# We fit t-SNE on the PCA-reduced features to speed up computation and reduce noise
tsne_results = tsne.fit_transform(reduced_features)

print("Plotting the t-SNE manifold...")

plt.figure(figsize=(10, 8))

# Create a colormap for the 6 clusters
colors = cm.rainbow(np.linspace(0, 1, n_clusters))

for cluster_id in range(n_clusters):
    # Select coordinates for points belonging to the current cluster
    indices = np.where(cluster_labels == cluster_id)[0]
    
    plt.scatter(
        tsne_results[indices, 0], 
        tsne_results[indices, 1], 
        color=colors[cluster_id], 
        label=f'Cluster {cluster_id}',
        alpha=0.7,
        edgecolors='w',
        s=50
    )

plt.title("t-SNE Visualization of Glomeruli Features", fontsize=16)
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# take the values of the first principal component (PC1) for all images
pc1_values = reduced_features[:, 0]

# Find the indices that would sort the PC1 values from lowest to highest
sorted_indices = np.argsort(pc1_values)

# Take the 5 extreme left (PC1 minimum) and 5 extreme right (PC1 maximum) indices
extreme_left_indices = sorted_indices[:5]
extreme_right_indices = sorted_indices[-5:]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("The First Principal Component (PC1) as a Measure of Disease Progression", fontsize=16)

# Plot the extreme left (Polo 1)
for i, idx in enumerate(extreme_left_indices):
    axes[0, i].imshow(X_data[idx])
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title(f"PC1 very Low\n{names_data[idx]}", fontsize=10)
    else:
        axes[0, i].set_title(f"{names_data[idx]}", fontsize=8)

# Plot the extreme right (Polo 2)
for i, idx in enumerate(extreme_right_indices):
    axes[1, i].imshow(X_data[idx])
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title(f"PC1 very High\n{names_data[idx]}", fontsize=10)
    else:
        axes[1, i].set_title(f"{names_data[idx]}", fontsize=8)

plt.tight_layout()
plt.show()

### Automated Cluster Ordering and Sequential t-SNE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print("Ordering clusters based on the main variance axis (PC1)...")

# 1. Calculate the mean PC1 value for each original cluster
cluster_means_pc1 = {}
for i in range(n_clusters):
    # reduced_features[:, 0] represents the First Principal Component (the main axis of variance)
    mean_pc1 = np.mean(reduced_features[cluster_labels == i, 0])
    cluster_means_pc1[i] = mean_pc1

# 2. Sort the original cluster IDs based on their mean PC1 value
# This orders them from one extreme of the biological continuum to the other
sorted_original_clusters = sorted(cluster_means_pc1, key=cluster_means_pc1.get, reverse=True)

# 3. Create a mapping dictionary: {old_id: new_sequential_id}
# new_sequential_id goes from 0 (e.g., Stage A) to 5 (e.g., Stage F)
cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted_original_clusters)}

# 4. Apply the mapping to create the new ordered labels
ordered_cluster_labels = np.array([cluster_mapping[label] for label in cluster_labels])

print("Mapping complete:")
for old_id, new_id in cluster_mapping.items():
    print(f"  - Original Cluster {old_id}  ->  Ordered Stage {new_id}")

# --- RE-PLOT t-SNE WITH SEQUENTIAL COLORS ---
print("\nPlotting sequential t-SNE...")
plt.figure(figsize=(10, 8))

# Use a sequential colormap (e.g., 'plasma' or 'viridis') to emphasize progression
sequential_colors = cm.plasma(np.linspace(0, 1, n_clusters))

for stage_id in range(n_clusters):
    # Select coordinates for points belonging to the current ordered stage
    indices = np.where(ordered_cluster_labels == stage_id)[0]
    
    plt.scatter(
        tsne_results[indices, 0], 
        tsne_results[indices, 1], 
        color=sequential_colors[stage_id], 
        label=f'Stage {stage_id}',
        alpha=0.7,
        edgecolors='w',
        s=50
    )

plt.title("Sequential t-SNE: Disease Progression Continuum", fontsize=16)
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(loc='best', title="Ordered Stages")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Optimal K Analysis (Elbow Method & Silhouette Score)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Define the range of k values to test (from 2 to 8)
k_values = range(2, 9)
inertia_values = []
silhouette_scores = []

print("Evaluating optimal number of clusters (k)...")

for k in k_values:
    # Initialize and fit K-Means
    kmeans_eval = KMeans(n_clusters=k, random_state=42, n_init=10)
    eval_labels = kmeans_eval.fit_predict(reduced_features)
    
    # Store Inertia for the Elbow Method
    inertia_values.append(kmeans_eval.inertia_)
    
    # Store Silhouette Score
    sil_score = silhouette_score(reduced_features, eval_labels)
    silhouette_scores.append(sil_score)
    
    print(f"Tested k={k} -> Inertia: {kmeans_eval.inertia_:.2f} | Silhouette: {sil_score:.4f}")

# --- PLOTTING THE RESULTS ---
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Inertia (Elbow Method) on the primary Y-axis (left)
color_inertia = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia (Sum of Squared Distances)', color=color_inertia, fontsize=12)
ax1.plot(k_values, inertia_values, marker='o', color=color_inertia, linewidth=2, label='Inertia')
ax1.tick_params(axis='y', labelcolor=color_inertia)
ax1.grid(True, linestyle='--', alpha=0.6)

# Create a secondary Y-axis (right) sharing the same X-axis
ax2 = ax1.twinx()

# Plot Silhouette Score on the secondary Y-axis
color_silhouette = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color_silhouette, fontsize=12)
ax2.plot(k_values, silhouette_scores, marker='s', color=color_silhouette, linewidth=2, label='Silhouette')
ax2.tick_params(axis='y', labelcolor=color_silhouette)

plt.title("Elbow Method and Silhouette Analysis for Optimal k", fontsize=16)
fig.tight_layout()
plt.show()

### Save results

In [ ]:
import pandas as pd
import os

# Create a DataFrame with filenames and their corresponding ordered labels
metadata_df = pd.DataFrame({
    'filename': names_data,
    'label': ordered_cluster_labels
})

# Save the DataFrame to a CSV file in the main directory
csv_output_path = "./Dataset_Glomeruli_All/pseudo_labels.csv"
metadata_df.to_csv(csv_output_path, index=False)

print(f"Metadata and pseudo-labels successfully saved to: {csv_output_path}")

In [ ]:
import pickle
import os

# Define output paths inside the dataset directory
dataset_dir = "./dataset_glomeruli_all"
kmeans_save_path = os.path.join(dataset_dir, "kmeans_model.pkl")
pca_save_path = os.path.join(dataset_dir, "pca_model.pkl")

# 1. Save the K-Means clustering model
print("Saving K-Means model...")
with open(kmeans_save_path, 'wb') as file:
    pickle.dump(kmeans, file)

# 2. Save the PCA dimensionality reduction model
# This is crucial because the K-Means model expects PCA-reduced features
print("Saving PCA model...")
with open(pca_save_path, 'wb') as file:
    pickle.dump(pca, file)

print("\nSuccessfully saved K-Means and PCA models to disk.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

# 1. Compute the linkage matrix using Ward's method (minimizes variance within clusters)
# This represents the hierarchical tree building phase (Lab slides 26-31)
print("Computing hierarchical linkage tree...")
linkage_matrix = linkage(reduced_features, method='ward')

# 2. Plot the Dendrogram (matching the visualization from slides 22 and 64)
plt.figure(figsize=(12, 6))
plt.title("Hierarchical Clustering Dendrogram (Truncated)", fontsize=16)
plt.xlabel("Sample Index or (Cluster Size)", fontsize=12)
plt.ylabel("Distance (Threshold)", fontsize=12)

# Truncate the dendrogram to make it readable, showing only the last 30 merges
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90.,
    leaf_font_size=10.,
    show_contracted=True
)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 3. Apply Agglomerative Clustering by cutting the tree at k=6
n_clusters = 6
print(f"Cutting the tree to extract exactly {n_clusters} clusters...")
hierarchical_model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
hierarchical_labels = hierarchical_model.fit_predict(reduced_features)

# 4. Evaluate using Silhouette Score
hierarchical_sil_score = silhouette_score(reduced_features, hierarchical_labels)
print("\n--- Evaluation Comparison ---")
print(f"K-Means Silhouette Score (k=6): {sil_score:.4f}")
print(f"Hierarchical Silhouette Score (k=6): {hierarchical_sil_score:.4f}")

# 5. Plot the t-SNE manifold with the new Hierarchical labels
plt.figure(figsize=(10, 8))
colors = plt.cm.rainbow(np.linspace(0, 1, n_clusters))

for cluster_id in range(n_clusters):
    indices = np.where(hierarchical_labels == cluster_id)[0]
    plt.scatter(
        tsne_results[indices, 0], 
        tsne_results[indices, 1], 
        color=colors[cluster_id], 
        label=f'Hierarchical Cluster {cluster_id}',
        alpha=0.7,
        edgecolors='w',
        s=50
    )

plt.title("t-SNE Visualization with Hierarchical Clustering Labels", fontsize=16)
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()